# BIBE RFMI Image-to-Indices Demo

This notebook is organized as a six-part demonstration. Each part contains executable code, a short explanation, and the output produced from one synthetic open-eye child facial image.

The example image is synthetic and is not study data.

## Part 1. Environment Setup and Synthetic Input Image

This part imports the required Python libraries, optionally installs dependencies, locates the repository root, and displays the clean synthetic open-eye child image. The expected result is a printed project path and the starting image used in this public demo.

In [ ]:
from pathlib import Path
import subprocess
import sys

INSTALL = False

# Optional manual override. Use this only if automatic root detection fails.
# Example: ROOT_OVERRIDE = Path(r'C:\\path\\to\\mediapipe_child_face_analysis_RFMI')
ROOT_OVERRIDE = None

def looks_like_project_root(path):
    path = Path(path)
    return (path / 'requirements.txt').exists() and (path / 'scripts' / '01_prepare_project.py').exists()

def find_project_root():
    candidates = []
    if ROOT_OVERRIDE is not None:
        candidates.append(Path(ROOT_OVERRIDE))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    repo_names = ['mediapipe_child_face_analysis_RFMI', 'mediapipe_child_face_analysis_RFMI_repo_ready', 'jupyter_reproducible_workflow']
    for desktop_name in ['Desktop', '桌面']:
        desktop = Path.home() / desktop_name
        for repo_name in repo_names:
            candidates.append(desktop / repo_name)
        if desktop.exists():
            for repo_name in repo_names:
                candidates.extend(desktop.glob(f'**/{repo_name}'))
    for candidate in candidates:
        candidate = candidate.resolve()
        if looks_like_project_root(candidate):
            return candidate
    raise FileNotFoundError('Project root was not found. Set ROOT_OVERRIDE to the repository folder.')

ROOT = find_project_root()

if INSTALL:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt')], check=True)

import pandas as pd
from IPython.display import Image, display

OPEN_IMAGE = ROOT / 'example_data' / 'images' / 'SYN_open.jpg'
SUBJECT_ID = 'SYN'

print('Project root:', ROOT)
display(Image(filename=str(OPEN_IMAGE), width=430))

## Part 2. Manifest Creation

This part writes a manifest file for the synthetic open-eye example image. The expected output is `outputs_manifest/manifest.csv`, which records the subject identifier, image state, and copied analysis path.

In [ ]:
def run(args):
    cmd = [sys.executable] + [str(a) for a in args]
    print('RUN:', ' '.join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)

run([
    ROOT / 'scripts' / '01_prepare_project.py',
    '--root', ROOT,
    '--open-image', OPEN_IMAGE,
    '--child-id', SUBJECT_ID,
])

manifest = pd.read_csv(ROOT / 'outputs_manifest' / 'manifest.csv')
manifest

## Part 3. MediaPipe Landmark Detection

This part runs MediaPipe Face Landmarker and exports raw landmark coordinates. The expected outputs are `landmarks_raw.csv` with 478 coordinate rows for the example image and `detection_log.csv`.

In [ ]:
run([ROOT / 'scripts' / '02_detect_and_overlay.py', '--root', ROOT])

landmarks = pd.read_csv(ROOT / 'outputs_landmarks' / 'landmarks_raw.csv')
detection_log = pd.read_csv(ROOT / 'outputs_landmarks' / 'detection_log.csv')

print('Landmark rows:', len(landmarks))
display(detection_log)
landmarks.head()

## Part 4. Coordinate-Based Overlay Figures

This part displays three visual outputs: full-face landmarks, an eye-region zoom, and the RFMI distance-line overlay. These figures are generated from MediaPipe coordinates; no landmarks are manually placed.

In [ ]:
image_id = f'{SUBJECT_ID}_open'
overlay_paths = [
    ROOT / 'outputs_overlay' / 'full_face' / f'{image_id}_overlay.jpg',
    ROOT / 'outputs_overlay' / 'eye_zoom' / f'{image_id}_eye_zoom.jpg',
    ROOT / 'outputs_overlay' / 'rfmi_lines' / f'{image_id}_rfmi_lines.jpg',
]

for path in overlay_paths:
    print(path.name)
    display(Image(filename=str(path), width=650))

## Part 5. RFMI Index Calculation

This part converts selected official MediaPipe landmark coordinates into RFMI variables. The output table has one row per image or subject and one column per RFMI feature.

In [ ]:
run([ROOT / 'scripts' / '04_compute_rfmi.py', '--root', ROOT])

rfmi_image = pd.read_csv(ROOT / 'outputs_features' / 'rfmi_image_level.csv')
rfmi_subject = pd.read_csv(ROOT / 'outputs_features' / 'rfmi_subject_level.csv')

display(rfmi_image)
display(rfmi_subject)

## Part 6. Summary Tables and Output Checklist

This part creates the subject-level RFMI table and the descriptive summary table. It also lists the main output files so readers can quickly check what the workflow generated.

In [ ]:
run([ROOT / 'scripts' / '05_summarize_rfmi.py', '--root', ROOT])

subject_table = pd.read_csv(ROOT / 'outputs_stats' / 'tables' / 'rfmi_subject_indices.csv')
summary_table = pd.read_csv(ROOT / 'outputs_stats' / 'tables' / 'rfmi_summary.csv')

display(subject_table)
display(summary_table)

main_outputs = [
    ROOT / 'outputs_manifest' / 'manifest.csv',
    ROOT / 'outputs_landmarks' / 'landmarks_raw.csv',
    ROOT / 'outputs_landmarks' / 'detection_log.csv',
    ROOT / 'outputs_overlay' / 'full_face' / f'{image_id}_overlay.jpg',
    ROOT / 'outputs_overlay' / 'eye_zoom' / f'{image_id}_eye_zoom.jpg',
    ROOT / 'outputs_overlay' / 'rfmi_lines' / f'{image_id}_rfmi_lines.jpg',
    ROOT / 'outputs_features' / 'rfmi_image_level.csv',
    ROOT / 'outputs_features' / 'rfmi_subject_level.csv',
    ROOT / 'outputs_stats' / 'tables' / 'rfmi_subject_indices.csv',
    ROOT / 'outputs_stats' / 'tables' / 'rfmi_summary.csv',
]

for path in main_outputs:
    print(path.relative_to(ROOT), 'exists=', path.exists())